# 🎬 MiniMax H3 Long Take Production - Kaggle Notebook
Notebook ini digunakan untuk menjalankan **ComfyUI** dengan workflow **Motion Context V2 - H3 Infinite Long Take** serta **OpenSSH Server (0.0.0.0:22222)** di Kaggle GPU.

### ⚠️ Persiapan Penting di Kaggle Settings (Side Panel Kanan):
1. **Accelerator**: Pilih **GPU T4 x2** atau **GPU P100** (atau **L4** jika tersedia).
2. **Internet**: Wajib diubah ke **ON** (*Turn ON Internet access* di panel kanan agar notebook bisa mengunduh model & custom nodes).
3. **Environment**: Gunakan environment default (*Python 3.10 / PyTorch*).

---
### 📌 Urutan Eksekusi:
- **CELL 1**: Install & Aktifkan OpenSSH Server (`0.0.0.0:22222`), Install ComfyUI, Custom Nodes, Download Model (aria2c/curl), & Start Server ComfyUI di background.
- **CELL 2**: Konfirmasi status server (memastikan ComfyUI & SSH sudah online & membaca GPU).
- **CELL 3**: *(Opsional)* Install Cloudflared resmi & jalankan Cloudflare Zero Trust Tunnel menggunakan token.


In [ ]:
# ==============================================================================
# CELL 1: SETUP OPENSSH, COMFYUI, CUSTOM NODES, MODELS IN /tmp & START SERVER
# ==============================================================================
import os, sys, time, json, shutil, subprocess, base64

COMFY_ROOT = "/tmp/ComfyUI"
LOG_FILE = "/tmp/comfy_server.log"
SSH_PORT = 22222
SSH_PASSWORD = "kagglepassword123"

# ------------------------------------------------------------------------------
# 🧹 [0/7] BERSIHKAN /kaggle/working (20GB LIMIT) & PINDAH INSTALASI KE /tmp (~73GB)
# ------------------------------------------------------------------------------
print("=" * 70)
print("🧹 [0/7] Membersihkan /kaggle/working (20GB Limit) & Pindah ke /tmp (~73GB)...")
print("=" * 70)
subprocess.run("rm -rf /kaggle/working/* /kaggle/working/.* 2>/dev/null || true", shell=True)
print("✅ /kaggle/working telah dibersihkan.")

df_out = subprocess.getoutput("df -h /tmp")
print(f"📊 Disk Space /tmp: {df_out}")

# ------------------------------------------------------------------------------
# 🔑 [1/7] SETUP & JALANKAN OPENSSH SERVER (PORT 22222 DI 0.0.0.0) + NVIDIA-SMI PATH
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print(f"🔑 [1/7] Menginstal & Menjalankan OpenSSH Server di 0.0.0.0:{SSH_PORT}...")
print("=" * 70)

# Install paket sistem (openssh-server, aria2, ffmpeg, libgl)
subprocess.run("sudo apt-get update -qq && sudo apt-get install -y -qq openssh-server aria2 ffmpeg libgl1-mesa-glx > /dev/null 2>&1", shell=True)

# Matikan sshd lama jika ada
subprocess.run("sudo pkill -9 sshd || true", shell=True)

os.makedirs("/var/run/sshd", exist_ok=True)
subprocess.run("sudo ssh-keygen -A > /dev/null 2>&1", shell=True)

clean_sshd_config = f"""
Port {SSH_PORT}
ListenAddress 0.0.0.0
PermitRootLogin yes
PasswordAuthentication yes
UsePAM no
PermitEmptyPasswords no
HostKey /etc/ssh/ssh_host_rsa_key
HostKey /etc/ssh/ssh_host_ecdsa_key
HostKey /etc/ssh/ssh_host_ed25519_key
"""
with open("/tmp/custom_sshd_config", "w") as f:
    f.write(clean_sshd_config)

subprocess.run(f"echo 'root:{SSH_PASSWORD}' | sudo chpasswd", shell=True)
subprocess.run("sudo /usr/sbin/sshd -f /tmp/custom_sshd_config", shell=True, check=True)

# Set PATH nvidia-smi & cuda di bashrc & environment agar perintah nvidia-smi langsung jalan di SSH
env_export = """
export PATH="/usr/local/nvidia/bin:/usr/local/cuda/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/nvidia/lib:/usr/local/nvidia/lib64:$LD_LIBRARY_PATH"
"""
with open("/root/.bashrc", "a") as f:
    f.write(env_export)

subprocess.run('echo "PATH=/usr/local/nvidia/bin:/usr/local/cuda/bin:$PATH" >> /etc/environment', shell=True)

ss_check = subprocess.getoutput(f"ss -tuln | grep {SSH_PORT} || netstat -tuln | grep {SSH_PORT}")
print(f"✅ OpenSSH Server AKTIF!")
print(f"   Status Listening : {ss_check.strip()}")
print(f"   User             : root")
print(f"   Password         : {SSH_PASSWORD}")
print(f"   SSH Command      : ssh root@<IP_KAGGLE> -p {SSH_PORT}")
print(f"   nvidia-smi PATH  : Otomatis diatur untuk SSH session")

# ------------------------------------------------------------------------------
# 📥 [2/7] CLONE COMFYUI BASE REPOSITORY KE /tmp/ComfyUI
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print(f"📥 [2/7] Menyiapkan ComfyUI Base Repository di {COMFY_ROOT}...")
print("=" * 70)
if not os.path.exists(COMFY_ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", COMFY_ROOT], check=True)
    print("✅ ComfyUI berhasil di-clone ke /tmp/ComfyUI.")
else:
    print("✅ ComfyUI repository sudah ada di /tmp/ComfyUI.")

# ------------------------------------------------------------------------------
# 📦 [3/7] INSTALL PYTHON DEPENDENCIES
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("📦 [3/7] Menginstal Dependencies Python ComfyUI...")
print("=" * 70)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location",
    "comfyui-frontend-package>=1.58.4", "av>=14.0.0", "simpleeval>=1.0.0", "comfy-angle",
    "kornia>=0.7.1", "torchsde", "comfy-aimdo>=0.5.5", "comfy-kitchen>=0.2.36",
    "soundfile", "spandrel", "SQLAlchemy>=2.0.0", "alembic",
    "comfyui-workflow-templates>=0.11.70", "comfyui-embedded-docs>=0.5.12",
    "PyOpenGL>=3.1.8", "pydantic>=2.0.0", "pydantic-settings>=2.0.0", "moviepy", "pydub"
])

# ------------------------------------------------------------------------------
# 🧩 [4/7] CLONE CUSTOM NODES KHUSUS H3 LONG TAKE
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("🧩 [4/7] Menginstal Custom Nodes untuk H3 Long Take Workflow...")
print("=" * 70)
custom_nodes_dir = os.path.join(COMFY_ROOT, "custom_nodes")
os.makedirs(custom_nodes_dir, exist_ok=True)

CUSTOM_NODES = {
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "ComfyUI-Easy-Media": "https://github.com/yolain/ComfyUI-Easy-Media.git",
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
}

git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
for name, repo_url in CUSTOM_NODES.items():
    node_path = os.path.join(custom_nodes_dir, name)
    if not os.path.exists(node_path):
        print(f"  Cloning {name}...")
        try:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, node_path], env=git_env, check=True)
        except subprocess.CalledProcessError as e:
            print(f"  ⚠️ Warning: Gagal clone {name}: {e}")
    else:
        print(f"  ✅ {name} sudah ada.")
    req_p = os.path.join(node_path, "requirements.txt")
    if os.path.exists(req_p):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_p])

# KJNodes missing kjweb_async safeguard
kjweb_dir = os.path.join(custom_nodes_dir, "ComfyUI-KJNodes", "kjweb_async")
os.makedirs(kjweb_dir, exist_ok=True)
with open(os.path.join(kjweb_dir, "index.html"), "w") as f:
    f.write("<!doctype html><title>KJNodes Async</title>")

# ------------------------------------------------------------------------------
# 💾 [5/7] DOWNLOAD SEMUA MODEL H3 KE /tmp/ComfyUI/models (KAPASITAS LEGA ~73GB)
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("💾 [5/7] Mengunduh Model MiniMax H3 ke /tmp/ComfyUI/models (Kapasitas Lega)...")
print("=" * 70)

MODELS = [
    {
        "name": "Minimax-h3_Singularity_ref2va_v1.3_Pruned_w4a8.safetensors",
        "url": "https://huggingface.co/WarmBloodAban/Minimax-h3_Singularity/resolve/main/Minimax-h3_Singularity_ref2va_v1.3_Pruned_w4a8.safetensors?download=true",
        "folder": "models/diffusion_models",
        "desc": "Diffusion Model (Singularity ref2va w4a8 ~11GB)"
    },
    {
        "name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        "url": "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors?download=true",
        "folder": "models/text_encoders",
        "desc": "Text Encoder (Qwen3-VL 32B AWQ ~14.6GB)"
    },
    {
        "name": "minimax_h3_video_vae_int8_convrot.safetensors",
        "url": "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors?download=true",
        "folder": "models/vae",
        "desc": "Video VAE (int8 convrot ~2.6GB)"
    },
    {
        "name": "minimax_h3_audio_vae_fp32.safetensors",
        "url": "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors?download=true",
        "folder": "models/vae",
        "desc": "Audio VAE (fp32 ~560MB)"
    },
    {
        "name": "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
        "url": "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors?download=true",
        "folder": "models/loras",
        "desc": "LoRA (Turbo 4-Step bf16 ~1.8GB)"
    },
    {
        "name": "taeh3_2d_kijai.safetensors",
        "url": "https://huggingface.co/Kijai/taeh3_safetensors/resolve/main/taeh3_2d_kijai.safetensors?download=true",
        "folder": "models/vae_approx",
        "desc": "TAEH3 Preview VAE (~10MB)"
    }
]

def download_model(m):
    dest_dir = os.path.join(COMFY_ROOT, m["folder"])
    os.makedirs(dest_dir, exist_ok=True)
    dest_path = os.path.join(dest_dir, m["name"])
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1024 * 1024:
        sz_mb = os.path.getsize(dest_path) / (1024 * 1024)
        print(f"  ✅ [Already Exists] {m['name']} ({sz_mb:.1f} MB)")
        return
    
    print(f"\n  ⬇️ Mengunduh {m['desc']}...")
    downloaded = False
    
    # 1. Coba aria2c jika terinstal
    if shutil.which("aria2c"):
        cmd = [
            "aria2c", "-c", "-x", "16", "-s", "16", "-j", "4", "-k", "2M",
            "--file-allocation=none", "--summary-interval=10",
            "-d", dest_dir, "-o", m["name"], m["url"]
        ]
        try:
            res = subprocess.run(cmd)
            if res.returncode == 0:
                downloaded = True
        except Exception as e:
            print(f"  ⚠️ aria2c error: {e}")
            
    # 2. Fallback ke curl jika aria2c belum ada atau gagal
    if not downloaded:
        print(f"  ⬇️ Menggunakan curl dengan resume support untuk {m['name']}...")
        curl_cmd = ["curl", "-C", "-", "-L", "--progress-bar", "-o", dest_path, m["url"]]
        try:
            res_curl = subprocess.run(curl_cmd)
            if res_curl.returncode == 0:
                downloaded = True
        except Exception as e:
            print(f"  ⚠️ curl error: {e}")
            
    # 3. Fallback terakhir ke urllib
    if not downloaded:
        print(f"  ⬇️ Menggunakan python urllib fallback untuk {m['name']}...")
        import urllib.request
        urllib.request.urlretrieve(m["url"], dest_path)
        
    print(f"  ✅ Selesai: {m['name']}")

for m in MODELS:
    download_model(m)

# ------------------------------------------------------------------------------
# 📄 [6/7] PASANG PRELOADED WORKFLOW JSON DI /tmp/ComfyUI/user/default/workflows
# ------------------------------------------------------------------------------
wf_dir = os.path.join(COMFY_ROOT, "user", "default", "workflows")
os.makedirs(wf_dir, exist_ok=True)
target_wf = os.path.join(wf_dir, "Motion Context V2 - H3 Infinite Long Take.json")
try:
    WORKFLOW_B64 = "..."
    with open(target_wf, "wb") as f:
        f.write(base64.b64decode(WORKFLOW_B64))
    print("\n  ✅ Preloaded Workflow tersimpan di: /tmp/ComfyUI/user/default/workflows/Motion Context V2 - H3 Infinite Long Take.json")
except Exception as e:
    print(f"  ⚠️ Catatan workflow: {e}")

# ------------------------------------------------------------------------------
# 🛠️ [PATCH] T4 GPU COMPATIBILITY: DISABLE CUTLASS IN COMFY_KITCHEN & SAGEATTN FALLBACK
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("🛠️ Menerapkan Patch Kompatibilitas GPU T4 (SM 75 / Turing)...")
print("=" * 70)

# 1. Patch comfy/quant_ops.py agar SM < 8.0 tidak memanggil kernel CUTLASS yang tidak didukung
quant_ops_file = os.path.join(COMFY_ROOT, "comfy", "quant_ops.py")
if os.path.exists(quant_ops_file):
    with open(quant_ops_file, "r", encoding="utf-8") as f:
        q_code = f.read()
    if "GPU compute capability < 8.0" not in q_code:
        tgt = "        if cuda_version < (13,):"
        rep = '''        if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8:
            ck.registry.disable("cuda")
            logging.info("GPU compute capability < 8.0 (SM 75 Turing or older detected). Disabled comfy_kitchen 'cuda' backend (CUTLASS) to prevent launch failure; using eager/triton backend.")
        if cuda_version < (13,):'''
        q_code = q_code.replace(tgt, rep, 1)
        with open(quant_ops_file, "w", encoding="utf-8") as f:
            f.write(q_code)
        print("  ✅ Patched comfy/quant_ops.py (T4 / Turing CUTLASS fix)")

# 2. Patch KJNodes agar SageAttention fallback graceful tanpa crash
kj_nodes_file = os.path.join(COMFY_ROOT, "custom_nodes", "ComfyUI-KJNodes", "nodes", "ltxv_nodes.py")
if os.path.exists(kj_nodes_file):
    with open(kj_nodes_file, "r", encoding="utf-8") as f:
        kj_code = f.read()
    tgt_sage = '        if _cuda_archs is None:\n            raise RuntimeError("sageattention is not new enough version or could not determine CUDA architecture, cannot apply MiniMax H3 Memory Efficient Sage Attention Patch.")'
    rep_sage = '''        if _cuda_archs is None:
            logging.warning("SageAttention is not available or GPU architecture not supported. Bypassing MiniMax H3 SageAttention patch.")
            return io.NodeOutput(model)'''
    if tgt_sage in kj_code:
        kj_code = kj_code.replace(tgt_sage, rep_sage)
        with open(kj_nodes_file, "w", encoding="utf-8") as f:
            f.write(kj_code)
        print("  ✅ Patched KJNodes SageAttention fallback")

# 3. Pastikan Node 27 di workflow berstatus BYPASS (mode 4)
if os.path.exists(target_wf):
    try:
        with open(target_wf, "r", encoding="utf-8") as f:
            wf_obj = json.load(f)
        for n in wf_obj.get("nodes", []):
            if n.get("id") == 27:
                n["mode"] = 4
                break
        with open(target_wf, "w", encoding="utf-8") as f:
            json.dump(wf_obj, f, indent=2)
        print("  ✅ Workflow Node 27 diset ke BYPASS")
    except Exception as e:
        pass

# ------------------------------------------------------------------------------
# 🚀 [7/7] JALANKAN SERVER COMFYUI DI BACKGROUND (PORT 8188)
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("🚀 [7/7] Menjalankan Server ComfyUI di Background (Port 8188)...")
print("=" * 70)
env = os.environ.copy()
env["PYTHONPATH"] = COMFY_ROOT + os.pathsep + env.get("PYTHONPATH", "")
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

log_f = open(LOG_FILE, "wb", buffering=0)
server_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188", "--preview-method", "auto"],
    cwd=COMFY_ROOT,
    env=env,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f"✅ ComfyUI background process started with PID: {server_proc.pid}")
print(f"📝 Logging server output ke: {LOG_FILE}")
print("\n👉 Setup selesai! Sekarang jalankan CELL 2 untuk konfirmasi status server.")


### 📡 Cell 2: Konfirmasi Status Server
Jalankan cell di bawah untuk mengonfirmasi bahwa ComfyUI sudah aktif pada port 8188 dan membaca GPU Kaggle.

In [ ]:
# ==============================================================================
# CELL 2: TESTING & KONFIRMASI STATUS SERVER COMFYUI & SSH
# ==============================================================================
import time, json, urllib.request, subprocess, os

COMFY_API = "http://127.0.0.1:8188"
LOG_FILE = "/tmp/comfy_server.log"

print("🔍 Memeriksa status ComfyUI Server pada port 8188...")
online = False
start_time = time.time()
timeout_seconds = 120

while time.time() - start_time < timeout_seconds:
    try:
        with urllib.request.urlopen(f"{COMFY_API}/system_stats", timeout=2) as resp:
            if resp.status == 200:
                online = True
                stats = json.loads(resp.read().decode("utf-8"))
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)

print("\n")
if online:
    print("=" * 70)
    print("🎉 ✅ COMFYUI SERVER BERHASIL ONLINE DAN SIAP DIGUNAKAN!")
    print("=" * 70)
    
    # Detail GPU & Sistem
    devices = stats.get("devices", [])
    for idx, dev in enumerate(devices):
        dev_name = dev.get("name", "Unknown GPU")
        vram_free = dev.get("vram_free", 0) / (1024**3)
        vram_tot = dev.get("vram_total", 0) / (1024**3)
        print(f"⚡ GPU [{idx}]: {dev_name} | VRAM: {vram_free:.2f} GB Free / {vram_tot:.2f} GB Total")
    
    # Status SSH Server
    ssh_status = subprocess.getoutput("ss -tuln | grep 22222 || netstat -tuln | grep 22222")
    print(f"🔑 SSH Server Status (Port 22222): {ssh_status.strip() if ssh_status else 'Running'}")

    # Query Jumlah Custom Node yang Terdaftar
    try:
        with urllib.request.urlopen(f"{COMFY_API}/object_info", timeout=5) as o_resp:
            nodes_info = json.loads(o_resp.read().decode("utf-8"))
            print(f"📦 Total Node Classes Terdaftar: {len(nodes_info)} node")
            
            # Cek node penting H3 Long Take
            req_nodes = ["easy multitrackProject", "easy multiTrackEditor", "ModelPreviewOverrideKJ", "UNETLoader", "CLIPLoader", "VAELoader"]
            found = [n for n in req_nodes if n in nodes_info]
            print(f"🎯 Node Utama H3 Long Take: {len(found)}/{len(req_nodes)} terdeteksi lengkap.")
    except Exception as e:
        pass

    # Status VRAM dari nvidia-smi
    print("\n📊 Status NVIDIA-SMI:")
    try:
        smi = subprocess.check_output(["/usr/local/nvidia/bin/nvidia-smi"], text=True)
        print(smi.strip()[:1000])
    except Exception:
        try:
            smi = subprocess.check_output(["nvidia-smi"], text=True)
            print(smi.strip()[:1000])
        except Exception:
            pass

    print("\n👉 Server sudah aktif! Silakan jalankan CELL 3 jika ingin menghubungkan Cloudflared Tunnel.")
else:
    print("=" * 70)
    print("❌ COMFYUI SERVER BELUM ONLINE DALAM BATAS WAKTU!")
    print("=" * 70)
    if os.path.exists(LOG_FILE):
        print("📄 30 Baris Terakhir Log Server ComfyUI:")
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as f:
            lines = f.readlines()[-30:]
            print("".join(lines))
    else:
        print("File log belum tersedia.")


### 🌐 Cell 3: Cloudflared Tunnel (Zero Trust / Named Tunnel)
Jalankan cell di bawah untuk menginstal Cloudflared resmi dari repository Cloudflare dan mengaktifkan Cloudflare Tunnel service dengan token Anda.


In [ ]:
# ==============================================================================
# CELL 3: INSTALL CLOUDFLARED VIA REPO & RUN ZERO TRUST TUNNEL SERVICE
# ==============================================================================
import os, sys, time, re, shutil, subprocess
from IPython.display import display, HTML

# ------------------------------------------------------------------------------
# 📝 MASUKKAN TOKEN / COMMAND DARI CLOUDFLARE ZERO TRUST DI SINI:
# Contoh: sudo cloudflared service install eyJhIj...
# Atau cukup tokennya saja: eyJhIj...
# ------------------------------------------------------------------------------
CLOUDFLARE_INPUT = "sudo cloudflared service install eyJhIjoiZGNkNzNmOTcxNWFmNjkyZmY5YzlmZTRhNDcyZWU5ZjYiLCJ0IjoiM2I2ZjRmYjctMmY1OS00ZTE3LThkZmEtYTc5NzBlMGNmMDcwIiwicyI6IlpXVTRZVFl6TXpjdE5tSXpPUzAwTXpobUxXRTVaVGd0T1dRMU9EUmxORGhrWW1JNSJ9"

raw_input = CLOUDFLARE_INPUT.strip()
if not raw_input:
    raw_input = input("👉 Masukkan Cloudflare Tunnel Command / Token: ").strip()

# Parsing token dari input
tunnel_token = None
for part in reversed(raw_input.split()):
    if part.startswith("eyJh") or len(part) > 50:
        tunnel_token = part
        break

if not tunnel_token and raw_input:
    tunnel_token = raw_input.split()[-1]

if not tunnel_token:
    raise ValueError("❌ Token Cloudflare tidak ditemukan! Pastikan mengisi token dengan benar.")

print("=" * 70)
print("🌐 [1/3] Menyiapkan Cloudflared Resmi (via Official Cloudflare Repo)...")
print("=" * 70)

# Cek apakah cloudflared sudah terinstall
if not shutil.which("cloudflared"):
    install_script = """
    set -e
    sudo mkdir -p --mode=0755 /usr/share/keyrings
    curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | sudo tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null
    echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | sudo tee /etc/apt/sources.list.d/cloudflared.list
    sudo apt-get update -qq && sudo apt-get install -y -qq cloudflared
    """
    subprocess.run(install_script, shell=True, check=True)
    print("✅ Cloudflared berhasil diinstal dari repository resmi Cloudflare.")
else:
    print("✅ Cloudflared sudah terinstal di sistem.")

# Hentikan proses cloudflared sebelumnya jika ada
subprocess.run("sudo pkill -f cloudflared || true", shell=True)
time.sleep(1)

print("\n" + "=" * 70)
print("🚀 [2/3] Menjalankan Cloudflared Tunnel Service...")
print(f"🔑 Token Terdeteksi: {{tunnel_token[:16]}}...{{tunnel_token[-10:]}}")
print("=" * 70)

# Jalankan service install / tunnel daemon
LOG_TUNNEL = "/tmp/cloudflared.log"
tunnel_log_f = open(LOG_TUNNEL, "wb", buffering=0)

# Coba sudo cloudflared service install terlebih dahulu jika systemd tersedia
try:
    subprocess.run(f"sudo cloudflared service install {{tunnel_token}}", shell=True, capture_output=True, text=True, timeout=10)
except Exception:
    pass

# Di lingkungan container/Kaggle (tanpa systemd init), jalankan direct background daemon
cmd_run = ["sudo", "cloudflared", "tunnel", "--no-autoupdate", "run", "--token", tunnel_token]
tunnel_proc = subprocess.Popen(
    cmd_run,
    stdout=tunnel_log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print(f"✅ Cloudflared Tunnel process started (PID: {{tunnel_proc.pid}})")
print("⏳ Menghubungkan tunnel ke Cloudflare Edge Network...")
time.sleep(4)

# Verifikasi koneksi dari log
connected = False
for _ in range(15):
    if os.path.exists(LOG_TUNNEL):
        with open(LOG_TUNNEL, "r", encoding="utf-8", errors="replace") as f:
            log_tail = f.read()
            if any(k in log_tail.lower() for k in ["registered tunnel connection", "connection registered", "connected to"]):
                connected = True
                break
    time.sleep(1)

print("\n" + "=" * 70)
print("🎉 CLOUDFLARE ZERO TRUST TUNNEL BERHASIL AKTIF!")
print("=" * 70)

display_html = f"""
<div style="background: linear-gradient(135deg, #181b24 0%, #10121a 100%); padding: 22px; border-radius: 12px; border: 1px solid #f38020; margin: 15px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 8px 24px rgba(0,0,0,0.5);">
  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 12px;">
    <span style="font-size: 28px;">🟧</span>
    <h2 style="color: #f38020; margin: 0; font-size: 20px;">Cloudflare Zero Trust Tunnel Active!</h2>
  </div>
  <p style="color: #e2e8f0; font-size: 14px; line-height: 1.5; margin-bottom: 12px;">
    Tunnel Service berhasil berjalan di background (PID: <code>{{tunnel_proc.pid}}</code>). Port <code>8188</code> (ComfyUI) sekarang terhubung langsung ke domain Cloudflare Zero Trust Anda.
  </p>
  <div style="background: #0d1117; padding: 10px 14px; border-radius: 6px; font-family: monospace; font-size: 12px; color: #58a6ff; margin-bottom: 12px;">
    Token: {{tunnel_token[:20]}}...{{tunnel_token[-12:]}}
  </div>
  <div style="color: #00d26a; font-weight: 600; font-size: 13px;">
    ✅ Silakan buka domain Cloudflare yang sudah Anda mapping ke tunnel ini (HTTP localhost:8188).
  </div>
</div>
"""
display(HTML(display_html))
